# Train YOLOv8 Custom Object Counter and Export TFLite

Use this notebook in Google Colab with a GPU runtime.

Output files:
- `best.pt`: trained PyTorch checkpoint
- `*.tflite`: model for the Flutter app
- `data.yaml`: dataset class names used by the app labels

Recommended Colab menu: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
!pip install -U ultralytics roboflow pyyaml tensorflow -q

from ultralytics import YOLO
import os
import shutil
import yaml
from pathlib import Path

print('Setup complete')

## Configuration

Choose one dataset source:
- `roboflow`: download a Roboflow dataset version exported as YOLOv8.
- `drive_zip`: use a ZIP file in Google Drive. The ZIP must contain `data.yaml`, `train/`, `valid/` or `val/`, and optional `test/` folders.

In [ ]:
DATASET_SOURCE = 'roboflow'  # 'roboflow' or 'drive_zip'

# Training config
BASE_MODEL = 'yolov8n.pt'
PROJECT_NAME = 'object_counter'
RUN_NAME = 'yolov8n_custom'
IMG_SIZE = 640
EPOCHS = 100
BATCH = 16
PATIENCE = 20

# Roboflow config. Fill these if DATASET_SOURCE == 'roboflow'.
ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY'
ROBOFLOW_WORKSPACE = 'workspace-id'
ROBOFLOW_PROJECT = 'project-id'
ROBOFLOW_VERSION = 1

# Google Drive ZIP config. Fill this if DATASET_SOURCE == 'drive_zip'.
DRIVE_ZIP_PATH = '/content/drive/MyDrive/datasets/object_counter_yolov8.zip'
DRIVE_EXTRACT_DIR = '/content/object_counter_dataset'

EXPORT_HALF_FLOAT16 = True
EXPORT_INT8 = False

## Load Dataset

In [ ]:
def load_roboflow_dataset():
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download('yolov8')
    return str(Path(dataset.location) / 'data.yaml')

def load_drive_zip_dataset():
    from google.colab import drive
    drive.mount('/content/drive')
    extract_dir = Path(DRIVE_EXTRACT_DIR)
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(DRIVE_ZIP_PATH, extract_dir)
    candidates = list(extract_dir.rglob('data.yaml'))
    if not candidates:
        raise FileNotFoundError('data.yaml not found in dataset ZIP')
    return str(candidates[0])

if DATASET_SOURCE == 'roboflow':
    DATA_YAML = load_roboflow_dataset()
elif DATASET_SOURCE == 'drive_zip':
    DATA_YAML = load_drive_zip_dataset()
else:
    raise ValueError(f'Unknown DATASET_SOURCE: {DATASET_SOURCE}')

print('DATA_YAML =', DATA_YAML)
with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)
print(data_config)

## Train

In [ ]:
model = YOLO(BASE_MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    project=PROJECT_NAME,
    name=RUN_NAME,
    device=0,
)

RUN_DIR = Path(PROJECT_NAME) / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
print('BEST_PT =', BEST_PT)
assert BEST_PT.exists(), f'Missing trained checkpoint: {BEST_PT}'

## Validate

In [ ]:
trained_model = YOLO(str(BEST_PT))
metrics = trained_model.val(data=DATA_YAML, imgsz=IMG_SIZE, device=0)
print('mAP50-95:', metrics.box.map)
print('mAP50:', metrics.box.map50)

## Export TFLite / LiteRT

Recent Ultralytics versions use `format='litert'` for `.tflite` export. Older versions may still use `format='tflite'`; this cell tries LiteRT first and falls back to TFLite.

In [ ]:
def export_tflite(model, half=True, int8=False):
    export_kwargs = dict(imgsz=IMG_SIZE, half=half, int8=int8)
    if int8:
        export_kwargs['data'] = DATA_YAML
    try:
        return model.export(format='litert', **export_kwargs)
    except Exception as litert_error:
        print('LiteRT export failed, trying legacy TFLite export:')
        print(litert_error)
        return model.export(format='tflite', **export_kwargs)

exported_files = []
exported_files.append(export_tflite(trained_model, half=EXPORT_HALF_FLOAT16, int8=False))
if EXPORT_INT8:
    exported_files.append(export_tflite(trained_model, half=False, int8=True))

print('Exported files:')
for item in exported_files:
    print(item)

## Inspect TFLite Tensor Shapes

Use the printed output shape to confirm app decoder assumptions. For the current Flutter app, expected YOLO shape is usually `[1, 4 + num_classes, num_boxes]`.

In [ ]:
import tensorflow as tf

tflite_candidates = list(RUN_DIR.rglob('*.tflite'))
print('TFLite candidates:', tflite_candidates)
TFLITE_PATH = str(tflite_candidates[0])
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
print('Input details:')
print(interpreter.get_input_details())
print('Output details:')
print(interpreter.get_output_details())

## Download Files

After downloading the `.tflite` file, copy it into the Flutter project at `assets/models/`. If you keep the app's default path, name it `yolov8n_float16.tflite`.

In [ ]:
from google.colab import files

files.download(str(BEST_PT))
files.download(TFLITE_PATH)
files.download(DATA_YAML)